In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [30]:
class Source:
    "Источник изотропного излучения фотонов"
    def __init__(self, Y=10000):
        self.Y = Y

    def generate_photons(self, E, batch_size=100):
        N = np.random.poisson(self.Y * E)

        for start in range(0, N, batch_size):
            n = min(batch_size, N - start)

            phi = np.random.uniform(0, 2 * np.pi, n)
            cos_theta = np.random.uniform(-1, 1, n)
            sin_theta = np.sqrt(1 - cos_theta**2)

            yield np.array([
                sin_theta * np.cos(phi),
                sin_theta * np.sin(phi),
                cos_theta
            ])

In [ ]:
np.random.seed(42)
source = Source(Y=10000)

# Модифицирован генератор так, чтобы он возвращал (yield) по 100 фотонов вместо одного
batches = list(source.generate_photons(E=1.0))
N_photons = sum(batch.shape[1] for batch in batches)

print(f"Число фотонов в событии: {N_photons}")
print(f"Число пачек: {len(batches)}")
print(f"Форма первой пачки: {batches[0].shape}")
print(f"Форма последней пачки: {batches[-1].shape}")

Число фотонов в событии: 9964
Число пачек: 100
Форма первой пачки: (3, 100)
Форма последней пачки: (3, 64)


In [32]:
assert all(batch.shape[0] == 3 for batch in batches)
assert all(1 <= batch.shape[1] <= 100 for batch in batches)
assert all(np.allclose(np.linalg.norm(batch, axis=0), 1) for batch in batches)
print("Проверка размера пачек и нормировки направлений пройдена")

Проверка размера пачек и нормировки направлений пройдена


In [33]:
np.random.seed(43)
E_test = 0.02
n_events = 300
counts = np.array([
    sum(batch.shape[1] for batch in source.generate_photons(E_test))
    for _ in range(n_events)
])
lam = source.Y * E_test

print(f"Теоретическое среднее и дисперсия: {lam:.1f}")
print(f"Выборочное среднее: {counts.mean():.2f}")
print(f"Выборочная дисперсия: {counts.var(ddof=1):.2f}")

Теоретическое среднее и дисперсия: 200.0
Выборочное среднее: 200.19
Выборочная дисперсия: 220.36


In [ ]:
assert abs(counts.mean() - lam) < 5 * np.sqrt(lam / n_events)
assert sum(batch.shape[1] for batch in source.generate_photons(0.0)) == 0
print("Статистическая проверка пройдена")

Статистическая проверка пройдена
